# NASA C-MAPSS (FD002) Remaining Useful Life (RUL) 예측 — 실전 파이프라인 (좋은 점수 목표)

이 노트북은 **CMAPSS FD002** 데이터를 사용해  
**불러오기 → 변수 확인 → 전처리/피처엔지니어링 → 데이터 분할 → 시각화 → 모델링(딥러닝) → 학습 → 평가(RMSE + NASA Score)**  
까지 한 번에 실행 가능한 “실전형” 코드입니다.

> ✅ FD002 특징(공식 설명): **조건(operating conditions) 6개**, 고장모드 1개(HPC degradation),  
> train 엔진 260대 / test 엔진 259대 입니다.  
> (출처: NASA dataset 설명) citeturn0search0

---

## 0) 실행 전 준비

### 데이터 위치
Kaggle에서 내려받은 파일들이 아래처럼 있어야 합니다.

```
data/
  train_FD002.txt
  test_FD002.txt
  RUL_FD002.txt
```

경로가 다르면 아래 `DATA_DIR`만 바꿔주세요.

In [ ]:
# ====== 1) 라이브러리 ======
import os
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

In [ ]:
# ====== 2) 데이터 로드 ======
DATA_DIR = "data"   # <-- 필요시 변경

TRAIN_PATH = os.path.join(DATA_DIR, "train_FD002.txt")
TEST_PATH  = os.path.join(DATA_DIR, "test_FD002.txt")
RUL_PATH   = os.path.join(DATA_DIR, "RUL_FD002.txt")

assert os.path.exists(TRAIN_PATH), f"파일 없음: {TRAIN_PATH}"
assert os.path.exists(TEST_PATH),  f"파일 없음: {TEST_PATH}"
assert os.path.exists(RUL_PATH),   f"파일 없음: {RUL_PATH}"

# 원본 txt는 공백 구분 + 마지막에 빈 컬럼이 들어오는 경우가 많아 delim_whitespace=True 사용
train_raw = pd.read_csv(TRAIN_PATH, delim_whitespace=True, header=None)
test_raw  = pd.read_csv(TEST_PATH,  delim_whitespace=True, header=None)
rul_true  = pd.read_csv(RUL_PATH,   delim_whitespace=True, header=None, names=["RUL_true"])

print(train_raw.shape, test_raw.shape, rul_true.shape)
train_raw.head()

### 컬럼 이름 정의

CMAPSS 기본 포맷은 보통 다음 순서입니다.

- `unit` (엔진 ID)
- `cycle` (시간, 사이클)
- 운영조건 3개: `op1, op2, op3`
- 센서 21개: `s1 ~ s21`

(데이터에 따라 공백/빈 컬럼이 더 있을 수 있어 아래에서 자동 정리합니다.)

In [ ]:
# ====== 3) 컬럼 정리 ======
# 보통 26개(2 + 3 + 21)인데, 파일에 따라 마지막에 NaN 컬럼이 하나 더 붙는 경우가 있음
def fix_columns(df):
    df = df.copy()
    # 완전 비어있는 컬럼 제거
    df = df.dropna(axis=1, how="all")
    n = df.shape[1]
    if n < 26:
        raise ValueError(f"컬럼 수가 예상보다 적습니다: {n} (>=26 expected)")
    # 앞 26개만 사용(혹시 더 있어도 안전하게 컷)
    df = df.iloc[:, :26]
    cols = ["unit", "cycle", "op1", "op2", "op3"] + [f"s{i}" for i in range(1, 22)]
    df.columns = cols
    return df

train = fix_columns(train_raw)
test  = fix_columns(test_raw)

train.head()

## 4) RUL(Label) 만들기

Train은 엔진이 **고장날 때까지** 기록되어 있으므로  
각 엔진별 최대 cycle을 기준으로

`RUL = max_cycle(unit) - cycle`

로 라벨을 만듭니다.

또한 실전에서 자주 쓰는 트릭: **RUL을 상한(capping)**
- 초반 정상 구간은 RUL이 너무 커서 모델이 어려워질 수 있어,
- 예: 125 또는 130 등으로 잘라서(piecewise) 학습하면 성능이 좋아지는 경우가 많습니다.

In [ ]:
# ====== 4) Train RUL 생성 + cap ======
train = train.copy()

max_cycle = train.groupby("unit")["cycle"].max().rename("max_cycle")
train = train.merge(max_cycle, on="unit", how="left")
train["RUL"] = train["max_cycle"] - train["cycle"]
train.drop(columns=["max_cycle"], inplace=True)

RUL_CAP = 125
train["RUL_capped"] = train["RUL"].clip(upper=RUL_CAP)

train[["unit","cycle","RUL","RUL_capped"]].head()

## 5) 간단 EDA(시각화)

- 엔진 1대의 RUL 변화
- 센서 분포/상관

In [ ]:
# ====== 5) EDA ======
example_unit = train["unit"].iloc[0]
df_u = train[train["unit"]==example_unit]

plt.figure()
plt.plot(df_u["cycle"], df_u["RUL"], label="RUL")
plt.title(f"Example unit={example_unit} RUL over cycles")
plt.xlabel("cycle"); plt.ylabel("RUL"); plt.legend()
plt.show()

# 센서 분산 확인(상수에 가까운 센서는 제거 후보)
sensor_cols = [c for c in train.columns if c.startswith("s")]
sensor_var = train[sensor_cols].var().sort_values()
sensor_var.head(10)

In [ ]:
plt.figure(figsize=(8,4))
sensor_var.tail(15).plot(kind="bar")
plt.title("Top-variance sensors (train)")
plt.tight_layout()
plt.show()

## 6) 피처 선택(상수/저분산 센서 제거)

C-MAPSS는 **거의 변하지 않는 센서가 섞여** 있는 경우가 많습니다.  
저분산 센서는 학습을 방해할 수 있어 제거합니다.

- 기준: 분산이 아주 작은 센서 제거(예: 1e-6)

In [ ]:
# ====== 6) 저분산 센서 제거 ======
VAR_TH = 1e-6
keep_sensors = sensor_var[sensor_var > VAR_TH].index.tolist()
drop_sensors = [c for c in sensor_cols if c not in keep_sensors]

print("drop sensors:", drop_sensors)
print("keep sensors:", len(keep_sensors))

feature_cols = ["op1","op2","op3"] + keep_sensors
target_col = "RUL_capped"

## 7) Train/Validation 분할 (엔진 단위로)

시계열 데이터는 **행 단위 random split** 하면 누수가 생깁니다.  
반드시 **엔진(unit) 단위**로 분할합니다.

In [ ]:
# ====== 7) 엔진 단위 split ======
units = train["unit"].unique()
train_units, val_units = train_test_split(units, test_size=0.2, random_state=SEED)

train_df = train[train["unit"].isin(train_units)].copy()
val_df   = train[train["unit"].isin(val_units)].copy()

print("train units:", len(train_units), "val units:", len(val_units))
print("train rows:", train_df.shape, "val rows:", val_df.shape)

## 8) 스케일링(정규화)

딥러닝은 입력 스케일에 민감합니다.  
일반적으로는 `StandardScaler`를 train 기준으로 fit 후 val/test에 적용합니다.

> (FD002는 운영조건이 6개라 조건별 정규화가 더 좋아질 수 있지만,  
> 여기서는 구현 난이도를 낮추면서도 성능이 잘 나오는 **전역 스케일링 + 딥모델**로 갑니다.)

In [ ]:
# ====== 8) scaling ======
scaler = StandardScaler()
scaler.fit(train_df[feature_cols])

def apply_scaling(df):
    out = df.copy()
    out[feature_cols] = scaler.transform(out[feature_cols])
    return out

train_df_s = apply_scaling(train_df)
val_df_s   = apply_scaling(val_df)
test_s     = apply_scaling(test)

train_df_s.head()

## 9) 시퀀스(윈도우) 만들기

딥러닝 모델은 `(window_len, n_features)` 형태의 시퀀스를 입력으로 받습니다.

- window 길이: 30~50이 흔한 선택
- label: 윈도우 **마지막 시점**의 RUL(캡 적용)

또한 학습 데이터에서 모든 시점이 다 필요하지는 않으니
`step`을 조절해 데이터 수를 줄일 수도 있습니다. (기본 1)

In [ ]:
# ====== 9) sequence generator ======
WINDOW = 30
STEP = 1

def make_windows(df, window=30, step=1, is_train=True):
    X_list, y_list, uid_list = [], [], []
    for uid, g in df.groupby("unit"):
        g = g.sort_values("cycle")
        data = g[feature_cols].values
        if is_train:
            y = g[target_col].values
            for start in range(0, len(g)-window+1, step):
                end = start + window
                X_list.append(data[start:end])
                y_list.append(y[end-1])  # last timestep label
                uid_list.append(uid)
        else:
            # test: 엔진별 마지막 window 1개만 사용
            if len(g) >= window:
                X_list.append(data[-window:])
            else:
                # 짧으면 앞을 padding(첫 row 반복)
                pad = np.repeat(data[:1], window-len(g), axis=0)
                X_list.append(np.vstack([pad, data]))
            uid_list.append(uid)
    X = np.stack(X_list).astype(np.float32)
    if is_train:
        y = np.array(y_list).astype(np.float32)
        return X, y, np.array(uid_list)
    return X, np.array(uid_list)

X_train, y_train, uid_tr = make_windows(train_df_s, window=WINDOW, step=STEP, is_train=True)
X_val,   y_val,   uid_va = make_windows(val_df_s,   window=WINDOW, step=STEP, is_train=True)
X_test,  uid_te          = make_windows(test_s,     window=WINDOW, step=1,    is_train=False)

X_train.shape, X_val.shape, X_test.shape

## 10) 모델: Conv1D + BiLSTM (실전 성능 좋은 조합)

- Conv1D: 짧은 구간 패턴 추출(특징 자동학습)
- BiLSTM: 시간 방향 의존성 학습
- Dense: 최종 RUL 회귀

성능을 위해:
- Dropout
- BatchNorm
- EarlyStopping + ReduceLROnPlateau

In [ ]:
# ====== 10) 모델 정의 ======
n_features = X_train.shape[-1]

def build_model(window, n_feat):
    inp = keras.Input(shape=(window, n_feat))
    x = layers.Conv1D(64, kernel_size=3, padding="same", activation="relu")(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(64, kernel_size=3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)
    x = layers.Dropout(0.2)(x)

    # 간단 attention (가벼운 가중합)
    attn = layers.Dense(1, activation="tanh")(x)
    attn = layers.Flatten()(attn)
    attn = layers.Activation("softmax")(attn)
    attn = layers.RepeatVector(128)(attn)  # 2*64
    attn = layers.Permute([2,1])(attn)
    x = layers.Multiply()([x, attn])
    x = layers.Lambda(lambda t: tf.reduce_sum(t, axis=1))(x)

    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(1)(x)

    model = keras.Model(inp, out)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="mse",
        metrics=[keras.metrics.RootMeanSquaredError(name="rmse"),
                 keras.metrics.MeanAbsoluteError(name="mae")]
    )
    return model

model = build_model(WINDOW, n_features)
model.summary()

In [ ]:
# ====== 11) 학습 ======
BATCH = 256
EPOCHS = 80

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_rmse", patience=10, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_rmse", factor=0.5, patience=5, min_lr=1e-5),
]

hist = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# 학습 곡선
plt.figure()
plt.plot(hist.history["rmse"], label="train_rmse")
plt.plot(hist.history["val_rmse"], label="val_rmse")
plt.title("RMSE over epochs")
plt.xlabel("epoch"); plt.ylabel("rmse"); plt.legend()
plt.show()

## 12) 평가 지표

### (1) RMSE / MAE
가장 기본적인 회귀 지표.

### (2) NASA scoring function (PHM 2008에서 널리 쓰이는 점수)
예측 오차가 **과소추정(너무 늦게 고장난다고 예측)** 인 경우에 더 큰 패널티를 주는 비대칭 점수입니다.  
(문헌/대회에서 자주 쓰이는 형태)

- d = y_pred - y_true  
- d < 0 (과소추정) : exp(-d/13) - 1  
- d >= 0 (과대추정) : exp(d/10) - 1  
총합이 작을수록 좋습니다.

(관련 설명 예시: CMAPSS RUL scoring 소개) citeturn0search22

In [ ]:
# ====== 12) 평가 함수 ======
def nasa_score(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    d = y_pred - y_true
    s = np.where(d < 0, np.exp(-d / 13.0) - 1.0, np.exp(d / 10.0) - 1.0)
    return np.sum(s)

# validation 예측
val_pred = model.predict(X_val, batch_size=1024).reshape(-1)

rmse = math.sqrt(mean_squared_error(y_val, val_pred))
mae  = mean_absolute_error(y_val, val_pred)
score = nasa_score(y_val, val_pred)

print("VAL RMSE:", rmse)
print("VAL MAE :", mae)
print("VAL NASA score:", score)

In [ ]:
# 예측 vs 실제
plt.figure()
plt.scatter(y_val, val_pred, s=8)
plt.xlabel("True RUL"); plt.ylabel("Pred RUL")
plt.title("Validation: True vs Pred")
plt.show()

plt.figure()
err = val_pred - y_val
plt.hist(err, bins=60)
plt.title("Validation error distribution (pred-true)")
plt.xlabel("error"); plt.ylabel("count")
plt.show()

## 13) Test 평가 (test_FD002 + RUL_FD002)

test set은 각 엔진이 고장 나기 **이전까지**만 주어지고,  
`RUL_FD002.txt`에 각 엔진의 실제 RUL(마지막 시점 기준)이 제공됩니다.

- 엔진별 마지막 window 1개로 예측 → RUL_true와 비교

In [ ]:
# ====== 13) Test 예측 ======
test_pred = model.predict(X_test, batch_size=1024).reshape(-1)

# 엔진 순서가 unit 오름차순이라는 가정은 하지 말고, uid_te 기준으로 정렬
pred_df = pd.DataFrame({"unit": uid_te, "RUL_pred": test_pred})
pred_df = pred_df.sort_values("unit").reset_index(drop=True)

true_df = pd.DataFrame({"unit": np.sort(test["unit"].unique()), "RUL_true": rul_true["RUL_true"].values})
test_eval = true_df.merge(pred_df, on="unit", how="left")

test_rmse = math.sqrt(mean_squared_error(test_eval["RUL_true"], test_eval["RUL_pred"]))
test_mae  = mean_absolute_error(test_eval["RUL_true"], test_eval["RUL_pred"])
test_score = nasa_score(test_eval["RUL_true"], test_eval["RUL_pred"])

print("TEST RMSE:", test_rmse)
print("TEST MAE :", test_mae)
print("TEST NASA score:", test_score)

test_eval.head()

In [ ]:
plt.figure()
plt.scatter(test_eval["RUL_true"], test_eval["RUL_pred"], s=10)
plt.xlabel("True RUL"); plt.ylabel("Pred RUL")
plt.title("Test: True vs Pred (FD002)")
plt.show()

## 14) 점수 더 올리는 팁 (선택)

아래는 점수 개선에 자주 도움이 되는 실전 옵션입니다.

1. **운영조건(op1~op3) 기반 정규화**
   - FD002는 operating condition이 6개라 조건별로 스케일을 따로 잡으면 성능이 좋아질 수 있음.
   - 흔한 방법: op settings로 k-means clustering 후 cluster별 scaler.

2. **윈도우 길이/step 튜닝**
   - WINDOW 30/40/50 비교
   - STEP 1/2로 데이터량 조절

3. **모델 튜닝**
   - LSTM 유닛 수 증가, CNN 필터 증가
   - Transformer Encoder 블록 추가

4. **Label 설정**
   - RUL_CAP 125 외에 130/100 등 시도